# 02 — SteelSense-BiLSTM v2 on SteelDefectX (6 classes, as in the paper)

Full revised protocol on the six-class SteelDefectX subset: held-out test set, five seeds, tabular and CNN baselines, order and binning ablations, interpretability evidence, and an end-to-end latency breakdown.

**Prerequisite:** run `00_Dataset_Integrity_Audit.ipynb` first.

In [1]:
import sys, json, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from config import DATASETS, ExperimentConfig, SplitConfig, RESULTS_DIR
import splits, pipeline, experiments, metrics
print("modules loaded")

modules loaded


### The dataset used here

The **six-class SteelDefectX subset as used in the manuscript** - Crazing, Inclusion,
Patches, Pitted surface, Rolled in scale, Scratches; 1631 images, split 1142 / 163 / 326.

Inclusion is 557 of 1631 images (34%), so accuracy alone is not reportable here and every
table below leads with macro-F1 and balanced accuracy (Reviewer 1, Q10).

### What these numbers do and do not support

The split is built exactly as in notebook 01 - image level, stratified, near-duplicate
groups pinned to one side - so **the results here are internally valid** as a standalone
benchmark, and every reviewer question about protocol is answered on this dataset in the
sections below.

What the overlap with NEU-DET (notebook 00, SS2) rules out is narrower, but it has to be
stated in the manuscript:

* this subset is **not an independent second corpus** - 64.1% of its images also appear in
  NEU-DET, so a NEU-DET result and a SteelDefectX result are not two independent pieces of
  evidence and must not be presented as mutual confirmation;
* **no cross-dataset transfer claim** is supportable - a model trained on NEU-DET and
  tested here would be scoring its own training images;
* the crazing F1 of 1.000 Reviewer 2 questioned has its explanation there, not in the
  descriptors.

Section 9b measures that overlap directly rather than leaving it to be inferred, which is
what turns the reviewer's suspicion into a reported number.

There are no annotations for SteelDefectX, so localization is evaluated on NEU-DET only
(notebook 01, section 9).

> **Appendix option.** For a genuinely NEU-disjoint evaluation, change the dataset in
> section 1 to `"steeldefectx"` - the full 24-class version with duplicates removed
> (3545 images, 19 classes at usable size). Everything downstream adapts automatically.

## 1. Configuration and the frozen split · Major Issue 2 (decision letter)

Every knob is declared in `src/config.py`. The split is loaded from the manifest written
by notebook 00 and its hash is verified on load.

In [2]:
cfg = ExperimentConfig(dataset="steeldefectx_paper")
cfg.train.epochs = 40
cfg.train.aug_variants = 3          # TRAIN SPLIT ONLY
cfg.seeds = [42, 1337, 2024, 7, 20250101]

df = splits.load_split("steeldefectx_paper", RESULTS_DIR / "splits")   # hash-verified
OUT = cfg.out_dir()
print(json.dumps(cfg.to_dict(), indent=1, default=str)[:1200])
print("\nsplit:", df.split.value_counts().to_dict())

{
 "dataset": "steeldefectx_paper",
 "seeds": [
  42,
  1337,
  2024,
  7,
  20250101
 ],
 "split": {
  "train": 0.7,
  "val": 0.1,
  "test": 0.2,
  "seed": 20240501,
  "dedupe": "group",
  "dhash_bits": 8,
  "dup_hamming_max": 2,
  "dup_corr_min": 0.98,
  "xdup_hamming_max": 6,
  "xdup_corr_min": 0.97,
  "group_same_class_only": true,
  "reuse_frozen_split": "C:\\Users\\anmol\\OneDrive\\Desktop\\Steel_Surface_Defect_NEU_DET-DATASET\\paper_results\\splits\\split_v1.csv"
 },
 "features": {
  "version": "v2",
  "img_size": 200,
  "glcm_levels": 32,
  "glcm_distances": [
   1,
   3
  ],
  "lbp_points": 8,
  "lbp_radius": 1
 },
 "bins": {
  "n_bins": 5,
  "strategy": "quantile",
  "oor_policy": "clamp"
 },
 "model": {
  "arch": "bilstm",
  "embed_dim": 96,
  "hidden_dim": 192,
  "num_layers": 2,
  "dropout": 0.3,
  "max_len": 160,
  "pooling": "all"
 },
 "train": {
  "epochs": 40,
  "batch_size": 32,
  "lr": 0.0008,
  "weight_decay": 0.0001,
  "label_smoothing": 0.06,
  "grad_clip": 1.0,
 

## 2. Descriptors → bins → prompt · Major Issue 5 (decision letter — feature-to-prompt reproducibility)

The bin edges are fitted on the **training rows only**. Validation and test are
transformed with those edges, and every value that falls outside the fitted range is
counted (Reviewer 1, Q8).

In [3]:
bundle = pipeline.build_bundle(df, cfg, verbose=True)

print(f"\nfeatures per image : {len(bundle.feature_names)}")
print(f"vocabulary         : {len(bundle.tokenizer)} tokens")
print(f"sequence length    : {bundle.tokenizer.max_len}")
print(f"train rows         : {len(bundle.train.y)} (from {len(bundle.train.paths)} images "
      f"x {cfg.train.aug_variants + 1} variants)")
print(f"val / test images  : {len(bundle.val.y)} / {len(bundle.test.y)}")

[features] cache hit feat_7188f745fde23cc0beca.npz  X=(4568, 92)
[pipeline] train: 1142 images -> 4568 rows, 92 features
[features] cache hit feat_a7c8ddcea46abb81061a.npz  X=(163, 92)
[pipeline] val: 163 images -> 163 rows, 92 features
[features] cache hit feat_44b8d61bde6c001b3852.npz  X=(326, 92)
[pipeline] test: 326 images -> 326 rows, 92 features
[pipeline] vocab=647 tokens, seq_len=94, bins=5/quantile
[pipeline] test out-of-range rate: 0.0667% of values (0.061 features/image)

features per image : 92
vocabulary         : 647 tokens
sequence length    : 94
train rows         : 4568 (from 1142 images x 4 variants)
val / test images  : 163 / 326


In [4]:
from prompt import readable_prompt

print("Example prompts (first 14 tokens of each):\n")
for i in [0, len(bundle.test.y) // 3, 2 * len(bundle.test.y) // 3]:
    cls = bundle.classes[bundle.test.y[i]]
    print(f"[{cls}]")
    print("  " + readable_prompt(bundle.test.tokens[i], 14) + "\n")

Example prompts (first 14 tokens of each):

[Crazing]
  cnt_area_cv=b4 cnt_area_max=b3 cnt_area_mean=b1 cnt_aspect_max=b2 cnt_aspect_mean=b2 cnt_circularity_mean=b3 cnt_coverage=b3 cnt_extent_mean=b1 cnt_fg_frac=b2 cnt_n=b4 cnt_n_large=b1 cnt_n_small=b4 cnt_n_tiny=b4 cnt_solidity_mean=b2 ...

[Inclusion]
  cnt_area_cv=b0 cnt_area_max=b0 cnt_area_mean=b0 cnt_aspect_max=b4 cnt_aspect_mean=b4 cnt_circularity_mean=b2 cnt_coverage=b0 cnt_extent_mean=b0 cnt_fg_frac=b0 cnt_n=b1 cnt_n_large=b1 cnt_n_small=b1 cnt_n_tiny=b1 cnt_solidity_mean=b1 ...

[Pitted surface]
  cnt_area_cv=b1 cnt_area_max=b4 cnt_area_mean=b4 cnt_aspect_max=b0 cnt_aspect_mean=b0 cnt_circularity_mean=b4 cnt_coverage=b3 cnt_extent_mean=b4 cnt_fg_frac=b4 cnt_n=b1 cnt_n_large=b0 cnt_n_small=b2 cnt_n_tiny=b1 cnt_solidity_mean=b4 ...



### Reviewer 1, Q8 · Major Issue 5 — out-of-range values at inference, and the discretization thresholds

**Policy.** Bin edges come from the training split, so out-of-range values at inference are
expected. The declared policy is `clamp`: the value is assigned to the first or last bin
and the occurrence is counted. The token stays in-vocabulary, so a numeric descriptor can
never produce `<unk>`. The alternative policy `oor_token` emits a dedicated
`<feature>=oorlo` / `=oorhi` symbol, and that symbol is registered in the vocabulary at fit
time — the tokenizer is built from the *schema* (every feature × every bin), not from
observed training rows, so no symbol is ever unseen.

**Rate.** Measured below, per split.

In [5]:
oor_rows = []
for split in ["train", "val", "test"]:
    r = bundle.oor_report[split]
    oor_rows.append({
        "split": split,
        "rows": r["n_rows_seen"],
        "value_checks": r["total_value_checks"],
        "out_of_range_values": r["n_out_of_range_values"],
        "rate_pct": r["overall_rate_pct"],
        "oor_features_per_image": r["mean_oor_features_per_image"],
        "unk_token_rate": r["unk_token_rate"],
    })
display(pd.DataFrame(oor_rows))

print("Features most often out of range on the TEST split:")
display(pd.DataFrame(bundle.oor_report["test"]["per_feature"]).T.head(10))

,split,rows,value_checks,out_of_range_values,rate_pct,oor_features_per_image,unk_token_rate
0,train,4568,420256,0,0.0000,0.0000,0.0000
1,val,163,14996,16,0.1067,0.0980,0.0000
2,test,326,29992,20,0.0667,0.0610,0.0000


Features most often out of range on the TEST split:


,below,above,rate_pct
glcm_d1_energy_range,0.0000,2.0000,0.6130
cnt_area_max,0.0000,1.0000,0.3070
cnt_area_mean,0.0000,1.0000,0.3070
cnt_coverage,0.0000,1.0000,0.3070
cnt_fg_frac,0.0000,1.0000,0.3070
col_profile_cv,0.0000,1.0000,0.3070
edge_orient_coherence,0.0000,1.0000,0.3070
fft_band1,0.0000,1.0000,0.3070
glcm_d1_homogeneity_range,0.0000,1.0000,0.3070
glcm_d3_ASM_range,0.0000,1.0000,0.3070


## 3. Main result — five seeds (Reviewer 1, Q9) · Major Issue 2 & 4 (decision letter)

Five seeds, one frozen split. Checkpoints are ranked by **validation macro-F1**; the
top-5 snapshot ensemble is formed from that ranking; the test split is scored once per
seed. Reported as mean ± SD, with a bootstrap 95% CI on the seed-averaged prediction.

This is the direct answer to Major Issue 2 (which subset trains, selects, and reports —
never the same one) and Major Issue 4 (mean ± SD over independent seeds, not a single
configuration) from the decision letter.

In [ ]:
main = experiments.run_seeds(bundle, cfg, verbose=True)

agg = main["aggregate"]
tbl = pd.DataFrame({
    m: {"mean": agg[m]["mean"], "sd": agg[m]["sd"], "min": agg[m]["min"], "max": agg[m]["max"]}
    for m in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
}).T
display(tbl)

sa = main["seed_averaged_prediction"]
print(f"\nSeed-averaged test accuracy : {sa['accuracy']:.4f} "
      f"(95% CI {sa['accuracy_ci95'][0]:.4f}-{sa['accuracy_ci95'][1]:.4f})")
print(f"Seed-averaged test macro-F1 : {sa['macro_f1']:.4f} "
      f"(95% CI {sa['macro_f1_ci95'][0]:.4f}-{sa['macro_f1_ci95'][1]:.4f})")
print(f"Test images: {main['n_test_images']}  ->  one image is "
      f"{100/main['n_test_images']:.3f} accuracy points")

[seed 42] training bilstm ...


### Reviewer 1, Q10 · Major Issue 4 — per-class precision / recall / F1, balanced accuracy, and the confusion matrix

In [ ]:
pc = main["per_class_over_seeds"]
rows = []
for cls, d in pc.items():
    rows.append({
        "class": cls, "support": d["support"],
        "precision": d["precision"]["mean"], "precision_sd": d["precision"]["sd"],
        "recall": d["recall"]["mean"], "recall_sd": d["recall"]["sd"],
        "f1": d["f1"]["mean"], "f1_sd": d["f1"]["sd"],
    })
per_class = pd.DataFrame(rows).sort_values("f1")
display(per_class)
print(f"\nWorst class by F1: {per_class.iloc[0]['class']} ({per_class.iloc[0]['f1']:.4f})")
print(f"Macro-F1 {agg['macro_f1']['mean']:.4f} vs accuracy {agg['accuracy']['mean']:.4f} "
      f"-- the gap is what accuracy alone would have hidden.")

In [ ]:
cm = np.array(sa["metrics"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(1.1 * len(bundle.classes) + 3, 1.0 * len(bundle.classes) + 2.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(bundle.classes))); ax.set_yticks(range(len(bundle.classes)))
ax.set_xticklabels(bundle.classes, rotation=60, ha="right", fontsize=8)
ax.set_yticklabels(bundle.classes, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"{cfg.dataset} - test confusion matrix (seed-averaged)")
thr = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > thr else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 4. Reviewer 1, Q4 · Major Issue 3 & 6 — tabular baselines on the raw un-discretized vector

These consume the **same descriptors** the prompt is built from, before discretization.
The only difference from SteelSense-BiLSTM is the representation, so the gap between them
is what the text-prompt idea is actually worth. Hyper-parameters are selected on
validation; test is scored once. Every value here is measured by this codebase, on this
split — none are taken from another publication (Major Issue 3).

In [ ]:
import baselines_tabular as BT

tab_res, tab_probs = BT.run_all(
    bundle.train.X, bundle.train.y,
    bundle.val.X, bundle.val.y,
    bundle.test.X, bundle.test.y,
    bundle.classes, seed=42, verbose=True,
)

rows = [{"model": k, "val_macro_f1": v.get("val_macro_f1"),
         "test_accuracy": v.get("accuracy"), "test_macro_f1": v.get("macro_f1"),
         "balanced_accuracy": v.get("balanced_accuracy"), "n_params": v.get("n_params")}
        for k, v in tab_res.items() if "macro_f1" in v]
display(pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False))

## 5. Reviewer 1, Q5 · Major Issue 3 & 7 — MobileNetV3 and ShuffleNetV2 under the same protocol

Same split manifest, same selection rule, same single touch of the test split.
Parameters, MACs and batch-size-1 latency are measured by `src/complexity.py` for every
model including ours, so the efficiency table is internally consistent.

> Reduce `CNN_EPOCHS` if you are running on CPU and want a quick pass; the value used for
> the reported numbers is recorded in the output JSON.

In [ ]:
import baselines_cnn as BC

CNN_EPOCHS = 20
CNN_MODELS = ["MobileNetV3-Small", "ShuffleNetV2-x0.5"]   # add "MobileNetV2", "ResNet18" if time allows

tr = df[df.split == "train"]; va = df[df.split == "val"]; te = df[df.split == "test"]
cls_idx = {c: i for i, c in enumerate(bundle.classes)}
to_y = lambda d: np.array([cls_idx[c] for c in d["class"]], dtype=np.int64)

cnn_res, cnn_probs = {}, {}
for name in CNN_MODELS:
    print(f"\n=== {name} ===")
    m, p = BC.train_backbone(
        name,
        tr["path"].tolist(), to_y(tr),
        va["path"].tolist(), to_y(va),
        te["path"].tolist(), to_y(te),
        bundle.classes, seed=42, epochs=CNN_EPOCHS, verbose=True,
    )
    cnn_res[name], cnn_probs[name] = m, p

In [ ]:
# Efficiency table: our model measured the same way as the CNNs
import torch
from complexity import profile_model, hardware
from model import build_model

# A freshly built copy: parameter count, MACs and latency depend on the
# architecture and the input shape, not on the trained weights.
ss_probe = build_model("bilstm", len(bundle.tokenizer), len(bundle.classes), cfg.model)
example = torch.from_numpy(bundle.test.ids[:1])
ss_complexity = profile_model("SteelSense-BiLSTM", ss_probe, example, is_token_model=True, repeats=100)

eff = [{
    "model": "SteelSense-BiLSTM",
    "input": "92 descriptor tokens",
    "params_M": round(ss_complexity["n_params"] / 1e6, 3),
    "size_MB": ss_complexity["size_mb"],
    "MMACs": ss_complexity["mmacs"],
    "latency_bs1_ms": ss_complexity["latency_bs1"]["median_ms"],
    "test_macro_f1": agg["macro_f1"]["mean"],
}]
for k, v in cnn_res.items():
    cx = v["complexity"]
    eff.append({
        "model": k + (" (ImageNet)" if v["pretrained"] else " (scratch)"),
        "input": f"3x{v['img_size']}x{v['img_size']} image",
        "params_M": round(cx["n_params"] / 1e6, 3),
        "size_MB": cx["size_mb"],
        "MMACs": cx["mmacs"],
        "latency_bs1_ms": cx["latency_bs1"]["median_ms"],
        "test_macro_f1": v["macro_f1"],
    })
efficiency = pd.DataFrame(eff)
display(efficiency)
print(json.dumps(hardware(), indent=1))
print("\nNOTE: the BiLSTM latency column is the CLASSIFIER ONLY. Section 9 measures the "
      "end-to-end cost including descriptor extraction, which is the number a deployment "
      "claim has to use.")

### Paired significance test against the strongest baseline (Reviewer 1, Q9) · Major Issue 3

McNemar's exact test on the identical test images, because the predictions are paired
per image. The exact binomial form is used rather than the χ² approximation since the
discordant count is small.

In [ ]:
all_base = {**tab_res, **cnn_res}
all_prob = {**tab_probs, **cnn_probs}
cmp = experiments.compare_to_baselines(main, all_base, all_prob, bundle.test.y, bundle.classes)

display(pd.DataFrame(cmp["comparison_table"]).sort_values("macro_f1", ascending=False))
print(f"\nStrongest baseline: {cmp['strongest_baseline']} "
      f"(macro-F1 {cmp['strongest_baseline_macro_f1']:.4f})")
print(f"SteelSense-BiLSTM  : macro-F1 {cmp['steelsense_macro_f1_mean']:.4f}")
print("\nMcNemar exact test vs strongest baseline:")
print(json.dumps(cmp["mcnemar_vs_strongest"], indent=2))

## 6. Reviewer 1, Q6 · Major Issue 6 — does the token order matter?

Four conditions, identical everywhere except the sequence encoder and the token order:

| | encoder | order |
|---|---|---|
| **A** | BiLSTM | canonical (deployed) |
| **B** | BiLSTM | one fixed random permutation, same for every sample |
| **C** | BiLSTM | a fresh permutation per sample — order information destroyed |
| **D** | DeepSets (position-wise MLP, permutation invariant) | canonical |

**How to read it.** A ≈ B means the specific order is arbitrary. A ≈ C means the model is
not using order at all. A ≈ D means the recurrent encoder buys nothing over an order-free
encoder of the same width — in which case the paper should say so and use the cheaper one.

In [ ]:
order_ab = experiments.run_order_ablation(df, cfg, seeds=cfg.seeds[:3], verbose=True)

rows = []
for k, v in order_ab.items():
    if not isinstance(v, dict) or "aggregate" not in v:
        continue
    rows.append({
        "condition": k, "encoder": v["arch"], "order": v["order"],
        "accuracy_mean": v["aggregate"]["accuracy"]["mean"],
        "accuracy_sd": v["aggregate"]["accuracy"]["sd"],
        "macro_f1_mean": v["aggregate"]["macro_f1"]["mean"],
        "macro_f1_sd": v["aggregate"]["macro_f1"]["sd"],
        "n_params": v["n_params"],
    })
display(pd.DataFrame(rows))
print(json.dumps(order_ab["verdict"], indent=2))
print("\nPaired tests vs condition A:")
for k, v in order_ab["paired_tests_vs_A"].items():
    print(f"  {k}: mean_diff {v.get('mean_diff', float('nan')):+.4f}  "
          f"t p={v.get('t_p_value')}  wilcoxon p={v.get('wilcoxon_p_value')}")

## 6b. Which component earns its parameters? · Major Issue 6 (decision letter) — Reviewer 1 Q1 · Reviewer 2 Q6 · Reviewer 3 Q7/Q8

Six conditions, everything else held fixed at the main run's split, seeds, epochs and
checkpoint-selection rule (validation macro-F1; test scored once per condition):

| Condition | What changes |
|---|---|
| `full_5snapshot_ensemble` | nothing — the deployed configuration |
| `no_ensemble_best_checkpoint` | the SAME fit, rescored with only its single best-validation checkpoint |
| `pool_attention_only` / `pool_max_only` / `pool_mean_only` | the pooled head keeps exactly one of the three views |
| `no_discretization_numeric` | the raw standardized descriptor vector (fit on **train only**), through the identical BiLSTM encoder and pooled head — no discretizer, no tokenizer anywhere in this condition |

The tabular baselines in §4 already show what a *different model family* (trees, a generic
MLP) does on the raw vector; they cannot isolate discretization on its own because they are
not the deployed encoder. `no_discretization_numeric` is the control that actually answers
"does converting continuous descriptors into categorical tokens throw away useful
information, or does it help" — with everything else about the architecture unchanged.

In [ ]:
ablation = experiments.run_component_ablation(
    bundle, cfg, seeds=cfg.seeds[:3], epochs=30, early_stop_patience=8, verbose=True,
)
abl_df = pd.DataFrame(ablation["summary"]).sort_values("macro_f1_mean", ascending=False)
display(abl_df)

print("\nPaired tests vs the full (deployed) configuration:")
for k, v in ablation["paired_tests_vs_full"].items():
    print(f"  {k:30s} mean_diff {v.get('mean_diff', float('nan')):+.4f}  "
          f"t p={v.get('t_p_value')}  wilcoxon p={v.get('wilcoxon_p_value')}")
print("\n" + ablation["note"])

## 7. Reviewer 1, Q7 · Major Issue 5 — bin count and edge rule

The submitted version used three semantic levels (Low / Medium / High) with no
justification. This sweep supplies one, or replaces it: bin counts crossed with quantile
against equal-width edges, under identical splits. This is the discretization-threshold
justification Major Issue 5 of the decision letter asks for.

The out-of-range column matters here too — equal-width bins on a skewed descriptor put
almost all mass in one bin and push more test values outside the fitted range.

**Compute budget.** A full 6 bin-counts x 2 strategies x 3 seeds grid at the main run's
epoch budget is `12 x 3 = 36` full trainings and was taking multiple days wall-clock on a
single CPU, with no way to resume after an interrupted kernel. The cell below trims the
grid to the bin counts that actually distinguish the answer (2 and 15 already showed the
plateau at both ends in an earlier run), drops to 2 seeds, and gives the sweep its own
lighter `epochs` / `early_stop_patience` / `batch_size` — legitimate here because this
is a relative comparison across bin configs, not the headline result, and checkpoint
selection still runs on validation macro-F1 only. `checkpoint_path` makes it resumable:
re-running the cell after an interruption skips every combo already finished instead of
starting over.

In [ ]:
sweep = experiments.run_bin_sweep(
    df, cfg, bin_counts=(3, 5, 7, 10),
    strategies=("quantile", "uniform"),
    seeds=cfg.seeds[:2],
    epochs=30, early_stop_patience=6, batch_size=64,
    checkpoint_path=OUT / "bin_sweep_checkpoint.json",
    verbose=True,
)
sw = pd.DataFrame(sweep["sweep"])
if "failed" in sw.columns:
    fails = sw[sw["failed"] == True]
    if len(fails):
        print("FAILED combos (see error column):")
        display(fails[["strategy", "n_bins", "error"]])
    sw = sw[sw["failed"] != True].reset_index(drop=True)
display(sw)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for strat, g in sw.groupby("strategy"):
    ax[0].errorbar(g.n_bins, g.macro_f1_mean, yerr=g.macro_f1_sd, marker="o", capsize=3, label=strat)
    ax[1].plot(g.n_bins, g.test_oor_rate_pct, marker="s", label=strat)
ax[0].set_xlabel("number of bins"); ax[0].set_ylabel("test macro-F1"); ax[0].legend()
ax[0].set_title(f"Bin count vs macro-F1 (mean +- SD, {len(cfg.seeds[:2])} seeds)"); ax[0].grid(alpha=.3)
ax[1].set_xlabel("number of bins"); ax[1].set_ylabel("test out-of-range rate (%)"); ax[1].legend()
ax[1].set_title("Bin count vs out-of-range rate"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("Best configuration:", json.dumps(sweep["best"], indent=2))


## 8. Reviewer 2, Q5 — substantiating "interpretable"

Two independent views, plus their agreement. Attention alone is not an explanation, so it
is checked against a causal measure.

* **Attention** — the pooling attention distribution, averaged per true class. Token
  position *i* is always feature *i*, so a weight is attributable to a named descriptor.
* **Permutation importance** — each descriptor's token is resampled across the test split
  and the drop in macro-F1 recorded over several repeats.
* **Agreement** — Spearman ρ between the two rankings. If ρ is low, the attention figure
  must not be presented as an explanation, and the manuscript should say so.

In [ ]:
import interpret

ss_model, ss_res = main["_model"]
attn = interpret.attention_by_class(
    ss_model, ss_res.snapshots, bundle.test.ids, bundle.test.y,
    bundle.feature_names, bundle.classes,
)

print("Most class-DISTINCTIVE descriptors (attention above the dataset mean):\n")
for cls, items in attn["most_distinctive_by_class"].items():
    top = ", ".join(f"{d['feature']}({d['delta_vs_mean']:+.4f})" for d in items[:4])
    print(f"  {cls:32s} {top}")

In [ ]:
perm = interpret.permutation_importance(
    ss_model, ss_res.snapshots, bundle.test.ids, bundle.test.y,
    bundle.feature_names, bundle.classes, n_repeats=3, verbose=True,
)
display(pd.DataFrame(perm["top20"]))

agree = interpret.agreement(attn, perm)
print(json.dumps(agree, indent=2))

In [ ]:
mean_attn = np.array(attn["mean_attention"])
top_idx = np.argsort(-mean_attn.mean(0))[:25]
fig, ax = plt.subplots(figsize=(13, 0.45 * len(bundle.classes) + 2.5))
im = ax.imshow(mean_attn[:, top_idx], aspect="auto", cmap="magma")
ax.set_yticks(range(len(bundle.classes))); ax.set_yticklabels(bundle.classes, fontsize=8)
ax.set_xticks(range(len(top_idx)))
ax.set_xticklabels([bundle.feature_names[j] for j in top_idx], rotation=75, ha="right", fontsize=7)
ax.set_title("Mean attention per class over the 25 most-attended descriptors")
plt.colorbar(im, fraction=0.02); plt.tight_layout(); plt.show()

## 9. Reviewer 2, Q6 · Major Issue 7 (decision letter) — per-stage latency at batch size 1

Every stage timed separately on the stated hardware, batch size 1, after warm-up.

**Read the totals before repeating any deployment claim.** Small in *parameters* is not the
same as fast at batch size 1: an LSTM is sequential over ~94 tokens, so its cost is set by
per-timestep kernel overhead rather than by its MAC count, and the 5-member snapshot
ensemble multiplies that by five. The order-free control from §6 is timed on the identical
input so the accuracy question and the cost question can be read together.

The thread-scaling measurement below replaces the unquantified "HPC-accelerated" claim
with a measured curve — report it or drop the claim.

In [ ]:
import profile_stages as PS

ds_probe = build_model("deepsets", len(bundle.tokenizer), len(bundle.classes), cfg.model)

prof = PS.profile_pipeline(
    bundle.test.paths[0], cfg.features, bundle.discretizer, bundle.tokenizer,
    ss_model, ss_res.snapshots, bundle.feature_names, deepsets_model=ds_probe,
    repeats=100, include_localization=False,
)

stages = pd.DataFrame(prof["stages_ms"]).T[["median_ms", "mean_ms", "sd_ms", "p95_ms"]]
stages["share_pct"] = (100 * stages["median_ms"] / prof["totals"]["end_to_end_ensemble_ms"]).round(2)
display(stages.sort_values("median_ms", ascending=False))
print(json.dumps(prof["totals"], indent=2))
print("\n" + prof["note"])

t = prof["totals"]
print(f"\nInference is {t['inference_share_of_end_to_end_pct']}% of end-to-end "
      f"cost at batch size 1.")
if "bilstm_vs_deepsets_speed_ratio" in t:
    print(f"The BiLSTM forward pass is {t['bilstm_vs_deepsets_speed_ratio']}x the cost of "
          f"the order-free control ({t['deepsets_forward_ms']} ms). Read this next to the "
          f"accuracy gap in section 6: if that gap is not significant, the recurrent "
          f"encoder is paying this ratio for nothing.")

In [ ]:
s = stages.sort_values("median_ms", ascending=True)
fig, ax = plt.subplots(figsize=(9, 0.38 * len(s) + 1.5))
ax.barh(s.index, s["median_ms"], color="steelblue", edgecolor="black")
ax.set_xlabel("median latency (ms), batch size 1")
ax.set_title(f"Per-stage cost — {prof['hardware']['processor'][:48]}")
for i, (name, v) in enumerate(s["median_ms"].items()):
    ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
ax.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

In [ ]:
speed = PS.speedup_report(bundle.test.paths[:120], cfg.features, thread_counts=(1, 2, 4, 8))
display(pd.DataFrame(speed["by_thread_count"]).T)
print("Report this curve, or remove the 'HPC-accelerated' claim from the manuscript.")

## 9b. The NEU-DET overlap, measured (Reviewer 1 Q2 · Reviewer 2 Q4)

The results above are a valid standalone benchmark on this dataset. This section addresses
the separate question the reviewers raised: **how much of this dataset is also NEU-DET, and
what does that do to the numbers?**

Two comparisons, both under the identical protocol and the same seeds:

| | dataset | what it isolates |
|---|---|---|
| **as reported** | the 6-class subset, 1631 images | the number in the paper |
| **NEU-disjoint** | the same subset with NEU-DET duplicates removed | what is left that is genuinely not NEU-DET |

The second row is expected to cover only part of the label space — de-duplication leaves
too few images in several of the six classes — and **that** is the finding to report:
the six-class SteelDefectX task cannot be reconstituted from NEU-disjoint data.

Stating this in the manuscript is what converts Reviewer 2's "invites rather than resolves
the leakage question" into a resolved question.

In [ ]:
overlap = {}
for ds in ["steeldefectx_paper", "steeldefectx_paper_clean"]:
    print(f"{ds} ...", flush=True)
    d2 = splits.build_split(ds, SplitConfig(), RESULTS_DIR / "splits", verbose=False)
    c2 = ExperimentConfig(dataset=ds)
    c2.train.epochs = cfg.train.epochs
    b2 = pipeline.build_bundle(d2, c2, verbose=False)
    r2 = experiments.run_seeds(b2, c2, seeds=cfg.seeds[:3], verbose=False)
    overlap[ds] = {
        "images": int(len(d2)),
        "classes": int(d2["class"].nunique()),
        "test_images": r2["n_test_images"],
        "accuracy": r2["aggregate"]["accuracy"]["mean"],
        "accuracy_sd": r2["aggregate"]["accuracy"]["sd"],
        "macro_f1": r2["aggregate"]["macro_f1"]["mean"],
        "macro_f1_sd": r2["aggregate"]["macro_f1"]["sd"],
        "balanced_accuracy": r2["aggregate"]["balanced_accuracy"]["mean"],
    }
    print(f"  acc {overlap[ds]['accuracy']:.4f}  macroF1 {overlap[ds]['macro_f1']:.4f}  "
          f"({overlap[ds]['classes']} classes, {overlap[ds]['images']} images)")

ov = pd.DataFrame(overlap).T
ov.index = ["6-class subset AS REPORTED", "same subset, NEU-DET duplicates removed"]
display(ov)

In [ ]:
# Per-class survival: which of the six classes exist at all without NEU-DET images
from duplicates import build_signatures, cross_dataset_duplicates

neu_i = splits.enumerate_images(DATASETS["neu_det"])
sdx_i = splits.enumerate_images(DATASETS["steeldefectx_paper"])
dups6 = cross_dataset_duplicates(
    build_signatures([i.path for i in sdx_i], 8),
    build_signatures([i.path for i in neu_i], 8),
    hamming_max=6, corr_min=0.97,
)
lab = {i.path: i.label for i in sdx_i}
n_by = pd.Series([i.label for i in sdx_i]).value_counts()
d_by = pd.Series([lab[p] for p in dups6]).value_counts()
surv = pd.DataFrame({"images": n_by, "also_in_NEU_DET": d_by}).fillna(0).astype(int)
surv["NEU_disjoint"] = surv["images"] - surv["also_in_NEU_DET"]
surv["pct_overlap"] = (100 * surv["also_in_NEU_DET"] / surv["images"]).round(1)
surv = surv.sort_values("pct_overlap", ascending=False)
display(surv)

lost = surv[surv["NEU_disjoint"] < 20]
print()
print(f"Overlap with NEU-DET: {surv['also_in_NEU_DET'].sum()} of {surv['images'].sum()} "
      f"images ({100*surv['also_in_NEU_DET'].sum()/surv['images'].sum():.1f}%)")
if len(lost):
    print(f"Classes with fewer than 20 NEU-disjoint images: {list(lost.index)}")
    print("The six-class task therefore cannot be reconstituted from NEU-disjoint data. "
          "Report this subset as a related benchmark, not as independent evidence.")

## 10. Persist everything

In [ ]:
payload = {
    "config": cfg.to_dict(),
    "split_counts": df.split.value_counts().to_dict(),
    "main_multiseed": main,
    "per_class": per_class.to_dict(orient="records"),
    "out_of_range": bundle.oor_report,
    "baselines_tabular": tab_res,
    "baselines_cnn": {k: {kk: vv for kk, vv in v.items()} for k, v in cnn_res.items()},
    "efficiency_table": efficiency.to_dict(orient="records"),
    "comparison_vs_baselines": cmp,
    "order_ablation": order_ab,
    "bin_sweep": sweep,
    "interpretability": {"attention": attn, "permutation_importance": perm, "agreement": agree},
    "stage_profile": prof,
    "thread_scaling": speed,
}

experiments.save_json(OUT / "results.json", payload)
efficiency.to_csv(OUT / "table_efficiency.csv", index=False)
per_class.to_csv(OUT / "table_per_class.csv", index=False)
pd.DataFrame(cmp["comparison_table"]).to_csv(OUT / "table_baselines.csv", index=False)
sw.to_csv(OUT / "table_bin_sweep.csv", index=False)
print("written to", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name)